In [26]:
from langchain_core.tools import tool
from dotenv import load_dotenv
load_dotenv()


@tool
def get_weather(location: str) -> str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

In [27]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0.7
)
model_with_tools = model.bind_tools([get_weather])

response = model_with_tools.invoke("What's the weather like in Boston?")
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to get weather. Use function get_weather with location "Boston".', 'tool_calls': [{'id': 'fc_55a7317d-03fd-4986-a75f-77b8391561a7', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 127, 'total_tokens': 170, 'completion_time': 0.091800189, 'completion_tokens_details': {'reasoning_tokens': 16}, 'prompt_time': 0.025011363, 'prompt_tokens_details': None, 'queue_time': 0.217342316, 'total_time': 0.116811552}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0a353-d0e7-70b0-bb30-517c09aec69d-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_55a7317d-03fd-4986-a75f-77b8391561a7', 'type': 'tool_call'}], invalid_tool_calls=[]

In [28]:
print(f"AI Message : {response.content}")
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call}")
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

AI Message : 
Tool: {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'fc_55a7317d-03fd-4986-a75f-77b8391561a7', 'type': 'tool_call'}
Tool: get_weather
Args: {'location': 'Boston'}


In [29]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)
print("AI Message without tool call :", messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

print("Manual Tool Calls : ", messages)
final_response = model_with_tools.invoke(messages)
print(final_response.text)

AI Message without tool call : [{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "What\'s the weather in Boston?" We need to call the get_weather function with location "Boston".', 'tool_calls': [{'id': 'fc_67f8626f-2506-4a6a-a0ca-aac0e35ddbfa', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 126, 'total_tokens': 177, 'completion_time': 0.106984528, 'completion_tokens_details': {'reasoning_tokens': 24}, 'prompt_time': 0.005191098, 'prompt_tokens_details': None, 'queue_time': 0.210928971, 'total_time': 0.112175626}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c15aa9c1b7', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0a353-d304-7560-8b87-c733ad2cf725-0', tool_calls=[{'name': 'get_we

In [38]:
@tool
def get_population(location: str) -> str:
    """Get the population of a location"""
    return f"{location} has about 700,000 residents"

# Map tool name -> tool function for easy lookup during execution
tools_map = {"get_weather": get_weather, "get_population": get_population}

# Bind BOTH tools (model may call them one at a time in sequential rounds)
model_with_tools = model.bind_tools([get_weather, get_population])

# Start conversation
messages = [{"role": "user", "content": "What's the weather and population of Boston?"}]


# Loop until the model stops calling tools and returns a final text response
while True:
    ai_msg = model_with_tools.invoke(messages)
    messages.append(ai_msg)

    # No more tool calls → model has finished
    if not ai_msg.tool_calls:
        break

    print(f"Model requested tool(s): {[tc['name'] for tc in ai_msg.tool_calls]}")

    # Execute each requested tool and append result as ToolMessage
    from langchain_core.messages import ToolMessage
    for tool_call in ai_msg.tool_calls:
        result = tools_map[tool_call["name"]].invoke(tool_call["args"])
        print(f"  -> {tool_call['name']}({tool_call['args']}) = {result}")
        messages.append(ToolMessage(content=result, tool_call_id=tool_call["id"]))

# Final answer from the model after all tools have been executed
print("\nFinal Answer:", ai_msg.content)


Model requested tool(s): ['get_weather']
  -> get_weather({'location': 'Boston'}) = It's sunny in Boston
Model requested tool(s): ['get_population']
  -> get_population({'location': 'Boston'}) = Boston has about 700,000 residents

Final Answer: **Boston, MA**

- **Current Weather:** Sunny  
- **Population:** Approximately 700,000 residents.


In [43]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.agents import create_agent

# Force the model to call a specific tool, every time
forced = model.bind_tools([get_weather], tool_choice="get_weather")
# Forbid tool calls entirely for this turn
no_tools = model.bind_tools([get_weather], tool_choice="none")
# Async tools -- @tool on an async def function
import asyncio
@tool
async def get_weather_async(location: str) -> str:
    """Get the weather at a location (calls an external API)."""
    await asyncio.sleep(0.2)
    return f"It's sunny in {location}"

async def main():
    agent = create_agent(model=ChatGroq(model="openai/gpt-oss-120b"), tools=[get_weather_async])
    response = await agent.ainvoke({"messages": [{"role": "user", "content": "Weather in Boston?"}]})
    print(response["messages"][-1].content)


await main()

# asyncio.run(main()) → Starts a new event loop, runs main(), and closes the loop. Best for standalone scripts.
# await main() → Runs inside an already running event loop. Best when you’re already inside async code (or in environments like Jupyter that keep an event loop alive).

It's sunny in Boston right now.
